# Lesson 2.1 — Prompt Engineering Principles: Clarity and Placement

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Write clear, specific instructions that reduce ambiguity
- Decide what belongs in the system prompt vs the user message
- Use structure (roles, sections, delimiters) to guide Claude
- Iteratively refine a weak prompt into a strong one

**You will produce:** A before/after prompt refinement harness with a reusable prompt template.

## Phase 2 Overview

Phase 2 covers **Domain 6 - Prompt and Context Engineering (11%)**: writing effective prompts, managing the context window to prevent drift and bloat, and producing/validating structured output. You will build the skills that make Claude behave predictably in production.

By the end of this phase, you will have built:

- A prompt-refinement harness
- A few-shot and output-constraint toolkit
- A context-window budget + compaction utility
- A subagent context-isolation pattern
- A structured-output validator with defensive parsing

### Prerequisites

- Completed Phase 1 (Messages API fundamentals)
- `pip install anthropic`
- Comfort reading JSON and writing Python functions

## Concepts

**Prompt engineering** is iteratively shaping instructions so Claude reliably produces the output you want. Core principles:

- **Clarity and specificity**: state the task, format, constraints, and audience explicitly. Ambiguity invites variance.
- **Placement**: durable role/behavior and global rules go in the **system** prompt; the specific task and data go in the **user** message. Keep instructions close to the content they govern.
- **Structure**: use headings, numbered steps, and delimiters (e.g., XML-style tags) to separate instructions from data - this also hardens against injection (Phase 5).
- **Iterative refinement**: test, inspect failures, and adjust one variable at a time.

> **Reference:** [Anthropic - Prompt engineering overview](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/overview)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — A vague prompt vs a specific one

Compare output quality between an underspecified and a well-specified prompt.

In [ ]:
def run(system, user, max_tokens=250):
    if not HAVE_API:
        return mock_text(user)
    msg = client.messages.create(model=MODEL_SONNET, max_tokens=max_tokens,
        system=system, messages=[{"role": "user", "content": user}])
    return msg.content[0].text

vague = "Tell me about this error."
specific = (
    "You are a debugging assistant. Given the stack trace below, respond with: "
    "(1) likely root cause in one sentence, (2) the exact line to inspect, "
    "(3) a suggested fix. Trace:\n<trace>KeyError: 'user_id' at app.py:42</trace>"
)
print("VAGUE:\n", run("", vague))
print("\nSPECIFIC:\n", run("You are precise and concise.", specific))

## Code-Along: Step 2 — System vs user placement

Put stable behavior in system; put the variable task in user.

In [ ]:
SYSTEM = (
    "You are a SQL reviewer. Always respond in exactly two sections: "
    "'Risks:' and 'Rewrite:'. Never execute SQL."
)
user_task = "Review: SELECT * FROM users WHERE name = '" + "'; DROP TABLE users;--"
print(run(SYSTEM, user_task))

## Exercise

Write `refine(prompt)` returning an improved version that (a) states a role, (b) specifies an output format, and (c) wraps user data in delimiters. Apply it to `'summarize this'`.

In [ ]:
def refine(task, data):
    # TODO: build a system string with a clear role + output format
    # TODO: wrap `data` in <data>...</data> delimiters in the user message
    # TODO: return (system, user)
    pass

## Key Takeaways

- Specific instructions (task, format, constraints, audience) reduce output variance.
- Put durable behavior in system; put the specific task and data in user messages.
- Delimiters and structure separate instructions from data and resist injection.
- Refine iteratively, changing one variable at a time.

---

**Next:** [Lesson 2.2 — Few-Shot Examples and Output Constraints](./lesson-2.2-few-shot-and-constraints.ipynb)